<a href="https://colab.research.google.com/github/nightclaw555/Comfy-colab/blob/main/colab/ComfyUI_A100_MiniMax_H3_Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 ComfyUI Video Studio — MiniMax-H3 Task Studio (A100 & L4 Auto-Adaptive)
### สตูดิโอผลิตวิดีโอพร้อมเสียงสเตอริโอด้วย MiniMax-H3 (รองรับทั้ง NVIDIA A100 และ L4 อัตโนมัติ)

> 💡 **One-Click Multi-Task MiniMax-H3 (Smart Multi-GPU Auto-Adaptive):**
> โน้ตบุ๊กนี้รองรับทั้ง **NVIDIA A100 (40GB/80GB)** และ **NVIDIA L4 (24GB)** โดยปรับระดับหน่วยความจำ (Dynamic VRAM Staging) ป้องกัน OOM 100% อัตโนมัติ!
> 1. **📱 TikTok Commercial Ads:** เรนเดอร์ความเร็วสูง 8-Step (~35s), สัดส่วน 9:16, เน้นขายสินค้าหรือนางแบบ
> 2. **💃 NSFW Motion Mimic:** ใช้โมเดล Hybrid (smhfacct) + MysticXXX คุมการเคลื่อนไหวตามคลิป Reference พร้อมเสียงตรงจังหวะ
> 3. **🎵 Cinematic Music Video:** งานภาพยนตร์คมชัดระดับมาสเตอร์พีซ รองรับ Multi-Shot ต่อเนื่องอัตโนมัติ


In [ ]:
# ====================================================================
# 🚀 ONE-CLICK A100 LAUNCHER — MINIMAX-H3 TURBO ONLY (FAST SSD MODE)
# ====================================================================
import os, sys, time, re, json, subprocess, torch, shutil

from google.colab import drive
print('⚡ [1/6] เชื่อมต่อ Google Drive และตรวจสอบ GPU...')
if not os.path.exists('/content/drive/MyDrive'):
    try: drive.mount('/content/drive')
    except Exception as e: print(f'Drive mount warning: {e}')

!nvidia-smi

# 1. ตรวจสอบการเชื่อมต่อ GPU อย่างปลอดภัย
if not torch.cuda.is_available():
    print("=" * 65)
    print("❌ ไม่พบ GPU ใน Runtime นี้! (ขณะนี้ระบบกำลังรันบน CPU)")
    print("👉 วิธีเชื่อมต่อ GPU บน Google Colab:")
    print("   1. ไปที่เมนูด้านบน: Runtime -> Change runtime type")
    print("   2. ตรง Hardware accelerator ให้เลือก 'GPU'")
    print("   3. ตรง GPU type ให้เลือก 'A100' หรือ 'L4' (แนะนำ L4 กรณี A100 คิวเต็ม)")
    print("   4. ตรง Runtime shape แนะนำเลือก 'High-RAM'")
    print("   5. กด 'Save' แล้วกดรันเซลล์นี้ใหม่อีกครั้งครับ")
    print("=" * 65)
    raise SystemExit("กรุณาเชื่อมต่อ GPU (A100 หรือ L4) ก่อนเริ่มรันระบบครับ")

gpu_name = torch.cuda.get_device_name(0)
gpu_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f'\n🔍 ตรวจพบการ์ดจอ: {gpu_name} (VRAM ทั้งหมด: {gpu_vram_gb:.1f} GB)')

# 2. ระบบ Smart Auto-Adaptive VRAM Staging รองรับทั้ง A100 80GB, A100 40GB, L4 24GB และอื่นๆ
if gpu_vram_gb >= 70.0:
    print(f'🚀 [ตรวจพบ {gpu_name} ~80GB]: เปิดประสิทธิภาพสูงสุด Full VRAM (--highvram --fast)')
    vram_flags = '--highvram --fast'
elif gpu_vram_gb >= 35.0:
    print(f'⚡ [ตรวจพบ {gpu_name} ~40GB]: รันความเร็วสูง ป้องกัน VRAM Peak (--reserve-vram 2.5 --fast)')
    vram_flags = '--reserve-vram 2.5 --fast'
elif gpu_vram_gb >= 20.0:
    print(f'🛡️ [ตรวจพบ {gpu_name} ~24GB]: เปิดโหมด Dynamic VRAM Staging สำหรับ L4 อัตโนมัติ!')
    print('   💡 ระบบจะสลับโมเดลระหว่าง High-RAM (51GB) และ GPU VRAM (24GB) ป้องกัน OOM 100% (--reserve-vram 1.5)')
    # บนการ์ด 24GB เช่น L4 ต้องไม่ใส่ --fast เพื่อให้ ComfyUI ทำ Dynamic CPU Offload ได้อย่างราบรื่น
    vram_flags = '--reserve-vram 1.5'
else:
    print(f'⚠️ [ตรวจพบ {gpu_name} < 20GB]: เปิดโหมด Low VRAM ป้องกันหน่วยความจำเต็ม (--lowvram)')
    vram_flags = '--lowvram --reserve-vram 1.0'

print('\n⏳ [2/6] ติดตั้ง ComfyUI + Custom Nodes (ข้ามการลง PyTorch ซ้ำเพื่อความเร็วสูงสุด)...')
!apt-get -y install aria2 > /dev/null 2>&1

if not os.path.exists('/content/ComfyUI'):
    %cd /content
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt
    !pip install -q fastapi uvicorn pydantic requests

    %cd /content/ComfyUI/custom_nodes
    !git clone --depth 1 https://github.com/Larryvrh/ComfyUI-MiniMax-H3-Turbo.git
    !git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
    !git clone --depth 1 https://github.com/kijai/ComfyUI-KJNodes.git
    !git clone --depth 1 https://github.com/Jalen-Brunson/ComfyUI-MiniMax-H3-PDD-Acc.git || true
    !pip install -q -r ComfyUI-VideoHelperSuite/requirements.txt || true

print('\n🚀 [3/6] ดึงไฟล์โมเดล MiniMax-H3 เข้า SSD ความเร็วสูงของ Colab โดยตรง...')
L_DIFF = '/content/ComfyUI/models/diffusion_models'
L_TEXT = '/content/ComfyUI/models/text_encoders'
L_VAE = '/content/ComfyUI/models/vae'
L_LORA = '/content/ComfyUI/models/loras'
L_OUT = '/content/ComfyUI/output'
for d in [L_DIFF, L_TEXT, L_VAE, L_LORA, L_OUT]:
    os.makedirs(d, exist_ok=True)

def dl_ssd(url, target_dir, filename):
    local_file = os.path.join(target_dir, filename)
    if os.path.exists(local_file) and os.path.getsize(local_file) > 1000000:
        print(f'⚡ [พร้อมใช้งานใน Local SSD] {filename}')
        return

    # ค้นหาใน Google Drive ก่อน (ถ้าเคยโหลดไว้แล้ว ดึงเข้า SSD ได้ใน 1 วินาที)
    subfolder = os.path.basename(target_dir)
    gdrive_paths = [
        os.path.join('/content/drive/MyDrive/ComfyUI_Studio/models', subfolder, filename),
        os.path.join('/content/drive/MyDrive/ComfyUI_Studio', filename),
        os.path.join('/content/drive/MyDrive/models', subfolder, filename),
        os.path.join('/content/drive/MyDrive', filename),
    ]
    for gp in gdrive_paths:
        if os.path.exists(gp) and os.path.getsize(gp) > 1000000:
            print(f'⚡ [พบใน Google Drive] เชื่อมต่อ {filename} เข้า High-Speed SSD ทันที...')
            try:
                if os.path.islink(local_file) or os.path.exists(local_file):
                    os.remove(local_file)
                os.symlink(gp, local_file)
                print(f'✅ เชื่อมต่อ Symlink สำเร็จใน 1 วินาที: {filename}')
                return
            except Exception:
                shutil.copy2(gp, local_file)
                print(f'✅ คัดลอกเข้า SSD สำเร็จ: {filename}')
                return

    # ถ้าไม่มีใน Drive ให้ดาวน์โหลดตรงเข้า SSD ด้วย aria2c
    print(f'📥 กำลังดาวน์โหลดเข้า SSD: {filename}...')
    !aria2c -c -x 16 -s 16 -k 1M --console-log-level=warn --summary-interval=5 "{url}" -d "{target_dir}" -o "{filename}"
    print(f'✅ โหลดสำเร็จ: {filename}')

# ====================================================================
# 🎛️ เลือกระดับโปรไฟล์ที่ต้องการรัน (แนะนำ: FULL_STUDIO โหลดครบจบใน A100)
# ====================================================================
STUDIO_PROFILE = 'FULL_STUDIO'  #@param ["FULL_STUDIO", "TIKTOK_FAST", "NSFW_MOTION", "MUSIC_VIDEO"]
print(f'📌 ใช้งานโปรไฟล์โมเดล: {STUDIO_PROFILE}')

# 1. Base Shared Essentials (Text Encoder, Audio/Video VAE, Turbo LoRA)
dl_ssd('https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors', L_TEXT, 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors')
dl_ssd('https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_video_vae_fp16.safetensors', L_VAE, 'minimax_h3_video_vae_fp16.safetensors')
dl_ssd('https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/vae/minimax_h3_audio_vae_fp32.safetensors', L_VAE, 'minimax_h3_audio_vae_fp32.safetensors')
dl_ssd('https://huggingface.co/larryvrh/MiniMax-H3-Turbo-Lora/resolve/main/minimax_h3_turbo_4step_ema_ckpt850.safetensors', L_LORA, 'minimax_h3_turbo_4step_ema_ckpt850.safetensors')

# 2. Diffusion Checkpoints ตามประเภทงาน
# A) TikTok & Music Video Base (FL2VA Pruned Int8 - คมชัดสูงมาก)
if STUDIO_PROFILE in ['FULL_STUDIO', 'TIKTOK_FAST', 'MUSIC_VIDEO']:
    dl_ssd('https://huggingface.co/Comfy-Org/MiniMax-H3/resolve/main/diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors', L_DIFF, 'minimax_h3_fl2va_pruned_int8_convrot.safetensors')

# B) NSFW & Motion Mimic Base (smhfacct Hybrid Model: FL2VA + REF2VA คุมท่าทางตามคลิป)
if STUDIO_PROFILE in ['FULL_STUDIO', 'NSFW_MOTION']:
    dl_ssd('https://huggingface.co/smhfacct/Minimax-H3-fl2va-ref2va-hybrid-models/resolve/main/minimax_h3_hybrid_fl2va_ref2va_b25-49-int8.safetensors', L_DIFF, 'minimax_h3_hybrid_fl2va_ref2va_b25-49-int8.safetensors')
    dl_ssd('https://huggingface.co/lynaNSFW/mysticxxx_MM_H3/resolve/main/MysticXXX_MMH3-V4.safetensors', L_LORA, 'MysticXXX_MMH3-V4.safetensors')

# C) Music Video PDD Acceleration (Parallel Decoding Distillation for Audio Sync)
if STUDIO_PROFILE in ['FULL_STUDIO', 'MUSIC_VIDEO']:
    dl_ssd('https://huggingface.co/aptech0081/MiniMax-H3-PDD-ComfyUI/resolve/main/minimax_h3_fl2va_pdd_acc_8step_comfyui.safetensors', L_LORA, 'minimax_h3_fl2va_pdd_acc_8step_comfyui.safetensors')

print('\n🛡️ [4/6] เริ่มต้น Sidecar Agent (Bulletproof Kill Switch & GPU Monitor)...')
tbe_addr = os.environ.get('TBE_RUNTIME_ADDR', '')
with open('/content/sidecar_config.json', 'w') as f:
    json.dump({'TBE_RUNTIME_ADDR': tbe_addr}, f)

# 🔗 ตั้งค่า Rclone Private API (ซิงค์วิดีโอเข้า G:\ComfyUI_Studio\outputs อัตโนมัติ ไม่ต้องกด Login ซ้ำ)
subprocess.run("apt-get -y install rclone > /dev/null 2>&1", shell=True, check=False)
with open("/content/rclone.conf", "w", encoding="utf-8") as rf:
    rf.write('[mygdrive]\ntype = drive\nclient_id = YOUR_GOOGLE_CLIENT_ID.apps.googleusercontent.com\nclient_secret = YOUR_GOOGLE_CLIENT_SECRET\nscope = drive\ntoken = {"access_token":"YOUR_GOOGLE_ACCESS_TOKEN","token_type":"Bearer","refresh_token":"YOUR_GOOGLE_REFRESH_TOKEN","expiry":"2026-11-05T00:55:49.1714138+07:00","expires_in":3599}\nteam_drive =' + "\n")

sidecar_code = r'''
"""
Colab Sidecar Agent for ComfyUI A100 Studio
- Monitors GPU VRAM, Utilization, and Temperature
- Monitors ComfyUI process status on port 8188
- Provides Auto-Disconnect / Kill Switch endpoint for Google Colab (Direct TBE Unassign + Fallback)
- Serves generated output metadata
"""

import os
import sys
import time
import json
import shutil
import threading
import subprocess
from typing import Optional
from pydantic import BaseModel
from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, FileResponse
import uvicorn

app = FastAPI(title="Colab Sidecar Agent", version="1.2.0")

# Enable CORS for Web App communication
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

START_TIME = time.time()
OUTPUT_DIR = os.environ.get("COMFY_OUTPUT_DIR", "/content/ComfyUI/output")
GDRIVE_DEST_DIR = os.environ.get("GDRIVE_OUTPUT_DIR", "/content/drive/MyDrive/ComfyUI_Studio/outputs")
RCLONE_CONF_PATH = os.environ.get("RCLONE_CONF_PATH", "/content/rclone.conf")
RCLONE_REMOTE_DEST = os.environ.get("RCLONE_REMOTE_DEST", "mygdrive:ComfyUI_Studio/outputs")
_SYNCED_FILES: dict[str, tuple[int, float]] = {}
_SYNC_LOCK = threading.Lock()


def sync_outputs_to_gdrive(force_all: bool = False):
    """Sync newly rendered videos/images from Colab SSD to Google Drive (both native Drive mount & Rclone)."""
    if not os.path.exists(OUTPUT_DIR):
        return

    with _SYNC_LOCK:
        # 1. Native Google Drive Mount Sync (/content/drive/MyDrive/ComfyUI_Studio/outputs)
        if os.path.exists("/content/drive/MyDrive"):
            try:
                os.makedirs(GDRIVE_DEST_DIR, exist_ok=True)
                for fname in os.listdir(OUTPUT_DIR):
                    if not fname.lower().endswith((".mp4", ".webm", ".mkv", ".png", ".jpg", ".jpeg", ".webp")):
                        continue
                    src_f = os.path.join(OUTPUT_DIR, fname)
                    dst_f = os.path.join(GDRIVE_DEST_DIR, fname)
                    if os.path.isfile(src_f):
                        src_stat = os.stat(src_f)
                        if src_stat.st_size > 1024 and (time.time() - src_stat.st_mtime) >= 1.0:
                            if not os.path.exists(dst_f) or os.stat(dst_f).st_size != src_stat.st_size:
                                shutil.copy2(src_f, dst_f)
                                print(f"[Google Drive Auto-Sync] Saved {fname} ({src_stat.st_size / (1024*1024):.2f} MB) -> {dst_f}")
            except Exception as e:
                print(f"[Google Drive Auto-Sync] Warning: {e}")

        # 2. Rclone Sync (if configured)
        if os.path.exists(RCLONE_CONF_PATH) and shutil.which("rclone"):
            try:
                if force_all:
                    subprocess.run(
                        [
                            "rclone", "copy", OUTPUT_DIR, RCLONE_REMOTE_DEST,
                            "--config", RCLONE_CONF_PATH,
                            "--transfers", "8",
                            "--drive-chunk-size", "128M",
                            "--include", "*.{mp4,webm,mkv,png,jpg,jpeg,webp}",
                            "--quiet"
                        ],
                        timeout=60,
                        check=False
                    )
                    return

                for fname in os.listdir(OUTPUT_DIR):
                    if not fname.lower().endswith((".mp4", ".webm", ".mkv", ".png", ".jpg", ".jpeg", ".webp")):
                        continue
                    fpath = os.path.join(OUTPUT_DIR, fname)
                    if not os.path.isfile(fpath):
                        continue
                    stat = os.stat(fpath)
                    if stat.st_size < 1024 or (time.time() - stat.st_mtime) < 1.5:
                        continue
                    sig = (stat.st_size, stat.st_mtime)
                    if _SYNCED_FILES.get(fname) == sig:
                        continue

                    res = subprocess.run(
                        [
                            "rclone", "copyto", fpath, f"{RCLONE_REMOTE_DEST}/{fname}",
                            "--config", RCLONE_CONF_PATH,
                            "--drive-chunk-size", "128M",
                            "--quiet"
                        ],
                        timeout=45,
                        check=False
                    )
                    if res.returncode == 0:
                        _SYNCED_FILES[fname] = sig
                        print(f"[Rclone Auto-Sync] Synced {fname} ({stat.st_size / (1024*1024):.2f} MB) -> {RCLONE_REMOTE_DEST}")
            except Exception as e:
                print(f"[Rclone Auto-Sync] Warning: {e}")


def _rclone_watcher_daemon():
    while True:
        time.sleep(3.0)
        sync_outputs_to_gdrive(force_all=False)


threading.Thread(target=_rclone_watcher_daemon, daemon=True).start()


def get_gpu_info():
    """Extract GPU information using nvidia-smi."""
    try:
        cmd = [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.used,memory.free,utilization.gpu,temperature.gpu",
            "--format=csv,noheader,nounits"
        ]
        res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, check=True)
        line = res.stdout.strip().split("
")[0]
        parts = [p.strip() for p in line.split(",")]
        return {
            "name": parts[0],
            "memory_total_mb": float(parts[1]),
            "memory_used_mb": float(parts[2]),
            "memory_free_mb": float(parts[3]),
            "utilization_gpu_pct": float(parts[4]),
            "temperature_c": float(parts[5]),
        }
    except Exception as e:
        return {
            "name": "GPU Unavailable or Not Found",
            "memory_total_mb": 0.0,
            "memory_used_mb": 0.0,
            "memory_free_mb": 0.0,
            "utilization_gpu_pct": 0.0,
            "temperature_c": 0.0,
            "error": str(e)
        }


def check_comfy_running(port: int = 8188) -> bool:
    """Check if ComfyUI is listening on port 8188."""
    import socket
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(1.0)
        return s.connect_ex(("127.0.0.1", port)) == 0


def find_tbe_addr() -> Optional[str]:
    """Find Colab internal runtime manager address."""
    # 1. From environment variable
    addr = os.environ.get("TBE_RUNTIME_ADDR")
    if addr:
        return addr

    # 2. From config file saved during notebook launch
    cfg_path = "/content/sidecar_config.json"
    if os.path.exists(cfg_path):
        try:
            with open(cfg_path, "r") as f:
                cfg = json.load(f)
                if cfg.get("TBE_RUNTIME_ADDR"):
                    return cfg["TBE_RUNTIME_ADDR"]
        except Exception:
            pass

    # 3. From /proc environment dumps of other processes (e.g. PID 1 or kernel)
    try:
        if os.path.exists("/proc"):
            pids = [1] + [int(p) for p in os.listdir("/proc") if p.isdigit() and p != "1"]
            for pid in pids:
                env_file = f"/proc/{pid}/environ"
                if os.path.exists(env_file):
                    try:
                        with open(env_file, "rb") as ef:
                            env_data = ef.read().decode("utf-8", errors="ignore")
                            for item in env_data.split(chr(0)):
                                if item.startswith("TBE_RUNTIME_ADDR="):
                                    val = item.split("=", 1)[1].strip()
                                    if val:
                                        return val
                    except Exception:
                        continue
    except Exception:
        pass

    return None


def perform_unassign():
    """
    Safely and completely unassign Google Colab runtime:
    1. Wait 1.5s so HTTP response reaches caller via Cloudflare.
    2. Direct HTTP POST to internal Colab unassign API (http://${TBE_RUNTIME_ADDR}/unassign),
       bypassing _output.eval_js which fails in standalone background processes.
    3. Terminate Cloudflare Tunnel & ComfyUI to immediately halt GPU usage and network traffic.
    4. Force container termination (kill PID 1) to ensure Colab releases the VM instance.
    """
    time.sleep(1.5)
    print("Flushing all rendered outputs to Google Drive via Rclone before unassign...")
    try:
        sync_outputs_to_gdrive(force_all=True)
    except Exception as e:
        print(f"Pre-unassign Rclone flush warning: {e}")

    print("Executing Google Colab runtime unassign...")

    # Step 1: Direct POST to Colab unassign API
    tbe_addr = find_tbe_addr()
    if tbe_addr:
        print(f"Calling Colab backend unassign at http://{tbe_addr}/unassign ...")
        try:
            import urllib.request
            req = urllib.request.Request(f"http://{tbe_addr}/unassign", data=b"", method="POST")
            with urllib.request.urlopen(req, timeout=5) as resp:
                print(f"Colab internal unassign endpoint returned status: {resp.status}")
        except Exception as e:
            print(f"Error calling internal unassign endpoint: {e}")
    else:
        print("TBE_RUNTIME_ADDR not found, proceeding to process and container termination...")

    # Step 2: Stop GPU compute & close network tunnels immediately
    print("Terminating cloudflared tunnel and ComfyUI...")
    try:
        subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False)
        subprocess.run(["fuser", "-k", "-9", "8188/tcp"], check=False)
        subprocess.run(["fuser", "-k", "-9", "8189/tcp"], check=False)
    except Exception as e:
        print(f"Error stopping services: {e}")

    # Step 3: Container termination fallback (kill PID 1)
    time.sleep(1.0)
    print("Terminating container PID 1 to ensure Colab releases VM...")
    try:
        os.system("kill -9 1 > /dev/null 2>&1 || kill -9 -1 > /dev/null 2>&1")
    except Exception as e:
        print(f"Error killing PID 1: {e}")
        os._exit(0)


@app.get("/")
@app.get("/health")
def health_check():
    """System health and quick overview."""
    gpu = get_gpu_info()
    uptime_sec = time.time() - START_TIME
    comfy_alive = check_comfy_running(8188)
    return {
        "status": "online",
        "uptime_seconds": round(uptime_sec, 1),
        "uptime_formatted": f"{int(uptime_sec // 3600)}h {int((uptime_sec % 3600) // 60)}m {int(uptime_sec % 60)}s",
        "gpu": gpu,
        "comfy_running": comfy_alive,
        "comfy_port": 8188
    }


@app.get("/stats")
def stats():
    """Comprehensive GPU and memory stats."""
    return get_gpu_info()


@app.post("/shutdown")
@app.post("/auto-disconnect")
def shutdown_colab(background_tasks: BackgroundTasks):
    """
    Auto-Disconnect Kill Switch:
    Immediately unassigns the Colab instance to prevent wasting Compute Units.
    """
    background_tasks.add_task(perform_unassign)
    return {
        "success": True,
        "message": "Shutdown signal accepted. Colab runtime will unassign in 2 seconds to preserve Compute Units.",
        "timestamp": time.time()
    }


@app.get("/outputs")
def list_outputs():
    """List generated video and image files in the outputs directory."""
    if not os.path.exists(OUTPUT_DIR):
        return {"outputs": [], "directory": OUTPUT_DIR}

    files = []
    for f in sorted(os.listdir(OUTPUT_DIR), reverse=True):
        full_path = os.path.join(OUTPUT_DIR, f)
        if os.path.isfile(full_path):
            stat = os.stat(full_path)
            files.append({
                "filename": f,
                "size_mb": round(stat.st_size / (1024 * 1024), 2),
                "modified": stat.st_mtime,
                "is_video": f.lower().endswith((".mp4", ".webm", ".mkv")),
                "is_image": f.lower().endswith((".png", ".jpg", ".jpeg", ".webp"))
            })
    return {"outputs": files, "count": len(files), "directory": OUTPUT_DIR}


@app.get("/outputs/{filename}")
def get_output_file(filename: str):
    """Serve output video or image file."""
    file_path = os.path.join(OUTPUT_DIR, filename)
    if not os.path.isfile(file_path):
        raise HTTPException(status_code=404, detail="File not found")
    return FileResponse(file_path)


class MergeVideosRequest(BaseModel):
    video1: Optional[str] = None
    video2: Optional[str] = None
    videos: Optional[list[str]] = None
    output_filename: Optional[str] = None


@app.post("/videos/merge")
def merge_videos(req: MergeVideosRequest):
    """
    Merge multiple 5-second videos seamlessly using FFmpeg concat.
    Supports either filenames residing in OUTPUT_DIR or relative paths.
    """
    if not os.path.exists(OUTPUT_DIR):
        raise HTTPException(status_code=400, detail="Outputs directory does not exist")

    def resolve_path(v: str) -> str:
        if os.path.isabs(v) and os.path.isfile(v):
            return v
        base_v = os.path.basename(v)
        p1 = os.path.join(OUTPUT_DIR, base_v)
        if os.path.isfile(p1):
            return p1
        p2 = os.path.join(OUTPUT_DIR, "video", base_v)
        if os.path.isfile(p2):
            return p2
        p3 = os.path.join("/content/drive/MyDrive/ComfyUI_Studio/outputs", base_v)
        if os.path.isfile(p3):
            return p3
        raise HTTPException(status_code=404, detail=f"Video file not found: {v}")

    video_list = req.videos if req.videos and len(req.videos) > 0 else ([req.video1, req.video2] if req.video1 and req.video2 else [])
    if len(video_list) < 2:
        raise HTTPException(status_code=400, detail="At least 2 videos are required for merging")

    paths = [resolve_path(v) for v in video_list]
    ts = int(time.time())
    out_name = req.output_filename or f"merged_{len(paths)*5}s_{ts}.mp4"
    if not out_name.endswith(".mp4"):
        out_name += ".mp4"
    out_path = os.path.join(OUTPUT_DIR, out_name)

    list_path = f"/content/merge_{ts}.txt"
    try:
        with open(list_path, "w") as f:
            for p in paths:
                f.write(f"file '{p}'
")

        cmd_fast = [
            "ffmpeg", "-y", "-f", "concat", "-safe", "0",
            "-i", list_path, "-c", "copy", out_path
        ]
        res = subprocess.run(cmd_fast, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        if res.returncode != 0 or not os.path.exists(out_path) or os.path.getsize(out_path) < 10000:
            inputs = []
            filter_parts = []
            for i, p in enumerate(paths):
                inputs.extend(["-i", p])
                filter_parts.append(f"[{i}:v][{i}:a]")
            concat_filter = "".join(filter_parts) + f"concat=n={len(paths)}:v=1:a=1[v][a]"
            cmd_filter = ["ffmpeg", "-y"] + inputs + [
                "-filter_complex", concat_filter,
                "-map", "[v]", "-map", "[a]",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "-c:a", "aac",
                out_path
            ]
            res_filter = subprocess.run(cmd_filter, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            if res_filter.returncode != 0:
                raise HTTPException(status_code=500, detail=f"FFmpeg merge error: {res_filter.stderr[-300:]}")
    finally:
        if os.path.exists(list_path):
            try: os.remove(list_path)
            except Exception: pass

    stat = os.stat(out_path)
    threading.Thread(target=sync_outputs_to_gdrive, kwargs={"force_all": True}, daemon=True).start()
    return {
        "success": True,
        "filename": out_name,
        "size_mb": round(stat.st_size / (1024 * 1024), 2),
        "url": f"/outputs/{out_name}",
        "message": f"Successfully merged {len(paths)} video clips seamlessly!"
    }


class ExtractFrameRequest(BaseModel):
    video: str
    output_filename: Optional[str] = None


@app.post("/videos/extract_last_frame")
def extract_last_frame(req: ExtractFrameRequest):
    """
    Extract the very last frame of a video using FFmpeg and save it as PNG
    to both ComfyUI input/ and output/ directories for direct sequential generation.
    """
    if not os.path.exists(OUTPUT_DIR):
        raise HTTPException(status_code=400, detail="Outputs directory does not exist")

    def resolve_path(v: str) -> str:
        if os.path.isabs(v) and os.path.isfile(v): return v
        base_v = os.path.basename(v)
        p1 = os.path.join(OUTPUT_DIR, base_v)
        if os.path.isfile(p1): return p1
        p2 = os.path.join(OUTPUT_DIR, "video", base_v)
        if os.path.isfile(p2): return p2
        raise HTTPException(status_code=404, detail=f"Video file not found: {v}")

    src_video = resolve_path(req.video)
    ts = int(time.time())
    out_name = req.output_filename or f"last_frame_{ts}.png"
    if not out_name.endswith(".png"):
        out_name += ".png"

    input_dir = "/content/ComfyUI/input"
    os.makedirs(input_dir, exist_ok=True)
    out_path_input = os.path.join(input_dir, out_name)
    out_path_output = os.path.join(OUTPUT_DIR, out_name)

    cmd = [
        "ffmpeg", "-y", "-sseof", "-0.1", "-i", src_video,
        "-update", "1", "-q:v", "2", out_path_input
    ]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if res.returncode != 0 or not os.path.exists(out_path_input):
        cmd_fallback = [
            "ffmpeg", "-y", "-i", src_video,
            "-vf", "select='eq(n\,last)'", "-vframes", "1", out_path_input
        ]
        subprocess.run(cmd_fallback, check=False)

    if os.path.exists(out_path_input):
        try:
            if os.path.abspath(out_path_input) != os.path.abspath(out_path_output):
                shutil.copyfile(out_path_input, out_path_output)
        except Exception:
            pass

    return {
        "success": True,
        "filename": out_name,
        "input_filename": out_name,
        "url": f"/outputs/{out_name}",
        "message": "Last frame extracted successfully!"
    }


COMFY_LOG_FILE = os.environ.get("COMFY_LOG_FILE", "/content/comfyui.log")


@app.get("/logs/comfyui")
def get_comfyui_logs(lines: int = 150):
    """Return the last N lines of ComfyUI console logs with error detection."""
    target_path = COMFY_LOG_FILE
    if not os.path.exists(target_path):
        if os.path.exists("comfyui.log"):
            target_path = "comfyui.log"
        elif os.path.exists("/content/sidecar.log"):
            target_path = "/content/sidecar.log"
        else:
            return {
                "exists": False,
                "logs": f"Log file not found at {COMFY_LOG_FILE}. ComfyUI may still be launching.",
                "total_lines": 0,
                "tail_count": 0,
                "has_error": False,
                "recent_errors": []
            }

    try:
        with open(target_path, "r", encoding="utf-8", errors="replace") as f:
            all_lines = f.readlines()

        tail_lines = all_lines[-lines:] if len(all_lines) > lines else all_lines
        log_text = "".join(tail_lines)

        recent_errors = []
        error_keywords = [
            "Traceback (most recent call last):",
            "OutOfMemoryError",
            "CUDA out of memory",
            "FileNotFoundError",
            "KeyError:",
            "RuntimeError:",
            "AttributeError:",
            "Error occurred when executing",
            "Exception:"
        ]
        for i, line in enumerate(tail_lines):
            for kw in error_keywords:
                if kw in line:
                    context = "".join(tail_lines[max(0, i - 1) : min(len(tail_lines), i + 8)]).strip()
                    if context not in recent_errors:
                        recent_errors.append(context)
                    break

        return {
            "exists": True,
            "logs": log_text,
            "total_lines": len(all_lines),
            "tail_count": len(tail_lines),
            "has_error": len(recent_errors) > 0,
            "recent_errors": recent_errors[-3:]
        }
    except Exception as e:
        return {
            "exists": False,
            "logs": f"Error reading log file: {str(e)}",
            "total_lines": 0,
            "tail_count": 0,
            "has_error": True,
            "recent_errors": [str(e)]
        }


if __name__ == "__main__":
    port = int(sys.argv[1]) if len(sys.argv) > 1 else 8189
    print(f"Starting Colab Sidecar Agent on port {port}...")
    uvicorn.run(app, host="0.0.0.0", port=port)

'''
with open('/content/colab_sidecar.py', 'w', encoding='utf-8') as f:
    f.write(sidecar_code)

# 🛡️ สตาร์ท Sidecar Agent ด้วย subprocess.Popen (Native Python ไม่บล็อก IPython Subshell 100%)
subprocess.run(["fuser", "-k", "-9", "8189/tcp"], check=False)
sidecar_env = os.environ.copy()
if tbe_addr:
    sidecar_env["TBE_RUNTIME_ADDR"] = tbe_addr
with open('/content/sidecar.log', 'w') as out_sidecar:
    subprocess.Popen(
        [sys.executable, "/content/colab_sidecar.py", "8189"],
        env=sidecar_env,
        stdin=subprocess.DEVNULL,
        stdout=out_sidecar,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )
print('✅ Sidecar Agent สตาร์ทเรียบร้อยแล้ว', flush=True)

print('\n🌐 [5/6] สตาร์ท ComfyUI (โหมด SSD ความเร็วสูง)...', flush=True)
subprocess.run(["fuser", "-k", "-9", "8188/tcp"], check=False)
comfy_env = os.environ.copy()
comfy_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.8"
comfy_args = [
    sys.executable, "main.py",
    "--listen", "0.0.0.0",
    "--port", "8188",
    "--output-directory", L_OUT
] + vram_flags.split()

with open('/content/comfyui.log', 'w') as out_comfy:
    subprocess.Popen(
        comfy_args,
        cwd="/content/ComfyUI",
        env=comfy_env,
        stdin=subprocess.DEVNULL,
        stdout=out_comfy,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )
print('✅ ComfyUI Core สตาร์ทเรียบร้อยแล้ว', flush=True)

print('\n🔗 [6/6] เปิด Cloudflare Named Tunnel เชื่อมต่อโดเมนถาวร...', flush=True)
TUNNEL_TOKEN = 'eyJhIjoiMjYwMTc0YTk0M2FkMDBmZDM3ZDc3NjQ4YjViY2ZkYzUiLCJ0IjoiZGVkMTU0ZjctNzEzNy00NjYyLWE1OTQtNjNiOWMyYWNiMGZlIiwicyI6Ik1UTTVaVGt3WkdJdFlUWm1NaTAwWkRoakxXSXdNRGd0T1RCa00yWmtZakE1TldVMiJ9'

if not os.path.exists('/usr/local/bin/cloudflared'):
    print('📥 กำลังติดตั้ง Cloudflared Client...', flush=True)
    subprocess.run("wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1", shell=True, check=False)

subprocess.run(["pkill", "-9", "-f", "cloudflared"], check=False)
if os.path.exists('/content/cf_tunnel.log'):
    try: os.remove('/content/cf_tunnel.log')
    except Exception: pass

with open('/content/cf_tunnel.log', 'w') as out_cf:
    subprocess.Popen(
        ["cloudflared", "tunnel", "run", "--token", TUNNEL_TOKEN],
        stdin=subprocess.DEVNULL,
        stdout=out_cf,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

print('⏳ กำลังรอให้พอร์ต 8188 และ 8189 ตอบสนอง...', flush=True)
import socket
for _ in range(15):
    time.sleep(1)
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s1, socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s2:
        s1.settimeout(0.5)
        s2.settimeout(0.5)
        if s1.connect_ex(('127.0.0.1', 8188)) == 0 and s2.connect_ex(('127.0.0.1', 8189)) == 0:
            break

comfy_url = 'https://comfy-api.ayaigaldie.site'
sidecar_url = 'https://sidecar.ayaigaldie.site'

print('=' * 65, flush=True)
print(f'🎉 เซิร์ฟเวอร์ MiniMax-H3 Turbo บน {gpu_name} เปิดเสร็จสมบูรณ์แล้ว!', flush=True)
print(f'⚡ การ์ดจอที่ใช้งาน: {gpu_name} ({gpu_vram_gb:.1f} GB VRAM) | Profile: {STUDIO_PROFILE}', flush=True)
print('💾 โมเดลทั้งหมดถูกเก็บใน : Local SSD (/content/ComfyUI/models)', flush=True)
print('📁 วิดีโอผลลัพธ์จะส่งตรงไปเซฟที่เครื่องและเบราว์เซอร์ของคุณอัตโนมัติ!', flush=True)
print(f'🔗 1. ComfyUI Endpoint URL : {comfy_url}', flush=True)
print(f'🔗 2. Sidecar Kill Switch URL: {sidecar_url}', flush=True)
print('=' * 65, flush=True)
